In [3]:
import sys
print(sys.version)

3.8.20 (default, Oct  3 2024, 15:24:27) 
[GCC 11.2.0]


In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService(channel="ibm_quantum",token="o0HHHmjs4XqpE1a8XStoNA1St_k1lzzoYx1LYxBgxnh-")
service.active_account()

RequestsApiError: 'HTTPSConnectionPool(host=\'auth.quantum-computing.ibm.com\', port=443): Max retries exceeded with url: /api/version (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7274e031a490>: Failed to resolve \'auth.quantum-computing.ibm.com\' ([Errno -2] Name or service not known)"))'

In [ ]:
import numpy as np
import pandas as pd

from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.units import DistanceUnit

# Bond lengths in Angstrom
bond_lengths = np.linspace(0.25, 2.0, 29)

energies = []

for r in bond_lengths:
    # Define H2 geometry
    geometry = f"H 0 0 0; H 0 0 {r}"

    # PySCF driver
    driver = PySCFDriver(
        atom=geometry,
        basis="sto3g",
        spin=0,
        charge=0,
        unit=DistanceUnit.ANGSTROM,
    )

    # Run electronic structure calculation
    problem = driver.run()

    # Ground-state energy (electronic + nuclear repulsion)
    energy = problem.reference_energy

    energies.append(energy)

# Create table
df = pd.DataFrame({
    "Bond Length (Å)": bond_lengths,
    "Energy (Hartree)": energies
})

print(df)
import matplotlib.pyplot as plt

plt.plot(bond_lengths, energies, marker='o')
plt.xlabel("Bond Length (Å)")
plt.ylabel("Energy (Hartree)")
plt.title("Classical: H₂ Ground-State Energy vs Bond Length (STO-3G)")
plt.grid(True)
plt.show()


In [ ]:
import numpy as np
import pandas as pd

from qiskit_algorithms import VQE
from qiskit_algorithms.optimizers import SLSQP
from qiskit.primitives import Estimator

from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.circuit.library import UCCSD, HartreeFock
from qiskit_nature.second_q.algorithms import GroundStateEigensolver
from qiskit_nature.units import DistanceUnit

bond_lengths = np.linspace(0.25, 2.00, 29)
vqe_energies = []

for R in bond_lengths:
    # --- Driver ---
    driver = PySCFDriver(
        atom=f"H 0 0 0; H 0 0 {R}",
        basis="sto3g",
        unit=DistanceUnit.ANGSTROM,
        charge=0,
        spin=0
    )

    problem = driver.run()

    # --- Mapper ---
    mapper = JordanWignerMapper()

    # --- Ansatz ---
    num_particles = problem.num_particles
    num_spatial_orbitals = problem.num_spatial_orbitals

    hf_state = HartreeFock(
        num_spatial_orbitals,
        num_particles,
        mapper
    )

    ansatz = UCCSD(
    num_spatial_orbitals=num_spatial_orbitals,
    num_particles=num_particles,
    initial_state=hf_state
    )
    ansatz.qubit_mapper = mapper


    # --- VQE ---
    optimizer = SLSQP(maxiter=1000)
    estimator = Estimator()

    vqe = VQE(
        estimator=estimator,
        ansatz=ansatz,
        optimizer=optimizer,
        initial_point=np.zeros(ansatz.num_parameters)
    )

    solver = GroundStateEigensolver(mapper, vqe)
    result = solver.solve(problem)

    vqe_energies.append(result.total_energies[0].real)

df = pd.DataFrame({
    "Bond Length (Å)": bond_lengths,
    "VQE (UCCSD) Energy (Ha)": vqe_energies
})

print(df)

import matplotlib.pyplot as plt

plt.plot(bond_lengths, vqe_energies, marker='o')
plt.xlabel("Bond Length (Å)")
plt.ylabel("Energy (Hartree)")
plt.title("VQE: H₂ Ground-State Energy vs Bond Length (STO-3G)")
plt.grid(True)
plt.show()
